# Assignment 1: Text Classification with Naive Bayes
**Natural Language Processing — Fall 2026**

This notebook follows the assignment requirements: preprocessing, a fixed train/test split, a vocabulary built from training data only, Bag of Words, visualizations, a Naive Bayes classifier implemented from scratch, comparison with scikit-learn, and evaluation.

**Dataset expected:** `synthetic_text_data.csv` with columns `text` and `label`.

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
)

# WordCloud is used only for the required visualizations.
try:
    from wordcloud import WordCloud
except ImportError:
    # Fallback for a fresh Colab/local kernel where requirements were not installed.
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wordcloud"])
    from wordcloud import WordCloud

RANDOM_SEED = 42
TEST_SIZE = 0.2
sns.set_theme(style="whitegrid")

## 1. Load the dataset

The repository includes `synthetic_text_data.csv` in its root directory. The notebook checks the repository root and `data/` folder automatically. If running in Google Colab without the repository files available, it will prompt you to upload the CSV. The dataset must contain the columns `text` and `label`.


In [ ]:
DATA_CANDIDATES = [
    "data/synthetic_text_data.csv",
    "synthetic_text_data.csv",
    "/content/synthetic_text_data.csv",
]

data_path = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)

if data_path is None:
    try:
        from google.colab import files

        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Please upload synthetic_text_data.csv.")
        data_path = csv_files[0]
    except ImportError:
        raise FileNotFoundError(
            "Place synthetic_text_data.csv in data/ or the notebook folder, then rerun this cell."
        )

raw_df = pd.read_csv(data_path)
required_columns = {"text", "label"}
if not required_columns.issubset(raw_df.columns):
    raise ValueError(
        f"CSV must contain columns {required_columns}; found {list(raw_df.columns)}"
    )

# Keep only the columns required by the assignment and report every removed row.
df = raw_df[["text", "label"]].copy()
raw_rows = len(df)
missing_rows = int(df.isna().any(axis=1).sum())
df = df.dropna().copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(str)
empty_rows = int(
    ((df["text"].str.strip() == "") | (df["label"].str.strip() == "")).sum()
)
df = df[
    df["text"].str.strip().ne("") & df["label"].str.strip().ne("")
].reset_index(drop=True)

print(f"Data file: {data_path}")
print(f"Raw rows: {raw_rows}")
print(f"Rows with missing text/label dropped: {missing_rows}")
print(f"Rows with empty text/label dropped: {empty_rows}")
print(f"Duplicate text rows kept: {int(df.duplicated(subset=['text']).sum())}")
print("Final dataset shape:", df.shape)
print("Classes:", sorted(df["label"].unique()))
display(df.head())
display(df["label"].value_counts().rename("count").to_frame())

## 2. Text preprocessing

Convert text to lowercase, extract word tokens, and remove punctuation by keeping alphabetic tokens. We keep the preprocessing simple and apply the same function to every sentence.

In [ ]:
def tokenize(text):
    """Lowercase text and return alphabetic word tokens."""
    text = str(text).lower()
    return re.findall(r"[a-z]+", text)

rows_before_preprocessing = len(df)
df["tokens"] = df["text"].apply(tokenize)
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))
df["sentence_length"] = df["tokens"].apply(len)

# A sentence with no alphabetic characters cannot contribute to the BoW model.
df = df[df["tokens"].map(len) > 0].reset_index(drop=True)

print(
    "Rows removed because they had no alphabetic tokens:",
    rows_before_preprocessing - len(df),
)
print("Rows after preprocessing:", len(df))
print("Empty token lists remaining:", int((df["tokens"].map(len) == 0).sum()))
display(df[["text", "clean_text", "label", "sentence_length"]].head())

## 3. Train/test split

Use a fixed random seed for reproducibility. The vocabulary and all word-based visualizations below are built from the training set only; the test set is held out for final evaluation.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df["label"],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

assert set(train_df["label"]) == set(test_df["label"]), (
    "Every class must be present in both splits."
)

print(f"Training rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print("\nTraining class distribution:")
display(train_df["label"].value_counts().rename("count").to_frame())


## 4. Data visualization

These plots use the training set to avoid using test examples for exploratory word analysis.

In [ ]:
# A. Most frequent words in training data
word_counts = Counter(token for tokens in train_df["tokens"] for token in tokens)
top_words = pd.DataFrame(word_counts.most_common(20), columns=["word", "frequency"])

plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=top_words, x="frequency", y="word", label="Training word count"
)
ax.legend(title="Training set", loc="lower right")
plt.title("Top 20 Word Frequencies (Training Set)")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

# B. Class distribution in training data
class_counts = (
    train_df["label"].value_counts().rename_axis("class").reset_index(name="count")
)
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=class_counts,
    x="class",
    y="count",
    label="Number of training sentences",
)
ax.legend(title="Training set", loc="upper right")
plt.title("Class Distribution (Training Set)")
plt.xlabel("Class")
plt.ylabel("Number of Sentences")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# C. Sentence length distribution in training data
plt.figure(figsize=(10, 5))
ax = sns.histplot(
    train_df["sentence_length"],
    bins=20,
    kde=False,
    label="Training sentences",
)
ax.legend(title="Training set", loc="upper right")
plt.title("Sentence Length Distribution (Training Set)")
plt.xlabel("Sentence Length (Number of Tokens)")
plt.ylabel("Number of Sentences")
plt.tight_layout()
plt.show()

# D. One word cloud per class, from training examples only
classes = sorted(train_df["label"].unique())
fig, axes = plt.subplots(
    len(classes), 1, figsize=(12, max(4, 4 * len(classes)))
)
if len(classes) == 1:
    axes = [axes]

for ax, label in zip(axes, classes):
    class_tokens = [
        token
        for tokens in train_df.loc[train_df["label"] == label, "tokens"]
        for token in tokens
    ]
    text_for_cloud = " ".join(class_tokens)
    if text_for_cloud.strip():
        cloud = WordCloud(
            width=1000,
            height=400,
            background_color="white",
            random_state=RANDOM_SEED,
        ).generate(text_for_cloud)
        ax.imshow(cloud, interpolation="bilinear")
        ax.set_title(f"Word Cloud — {label}")
    else:
        ax.text(0.5, 0.5, "No words available", ha="center", va="center")
        ax.set_title(f"Word Cloud — {label}")
    ax.axis("off")

legend_handle = Patch(
    facecolor="steelblue",
    label="Word size is proportional to its frequency in the class",
)
fig.legend(handles=[legend_handle], loc="lower center", frameon=False)
fig.suptitle("Word Clouds by Class (Training Set)")
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

## 5. Build a vocabulary and Bag of Words

The vocabulary is created **from the training tokens only**. Each sentence becomes a count vector: position *j* is the number of occurrences of vocabulary word *j* in that sentence.

In [ ]:
# The vocabulary is built from training tokens only, so test words cannot leak in.
vocabulary = sorted({token for tokens in train_df["tokens"] for token in tokens})
word_to_index = {word: i for i, word in enumerate(vocabulary)}


def tokens_to_bow(tokens, word_to_index):
    """Convert a token list to a fixed-length count vector using Counter."""
    counts = Counter(tokens)
    vector = np.zeros(len(word_to_index), dtype=np.int64)
    for word, count in counts.items():
        index = word_to_index.get(word)
        if index is not None:
            vector[index] = count
    return vector


X_train = np.vstack(
    [tokens_to_bow(tokens, word_to_index) for tokens in train_df["tokens"]]
)
X_test = np.vstack(
    [tokens_to_bow(tokens, word_to_index) for tokens in test_df["tokens"]]
)
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

# Test-only words are ignored by tokens_to_bow (the None branch).
oov_tokens = sorted(
    {
        token
        for tokens in test_df["tokens"]
        for token in tokens
        if token not in word_to_index
    }
)
empty_test_documents = int((X_test.sum(axis=1) == 0).sum())

print("Vocabulary size:", len(vocabulary))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Unseen test tokens ignored:", len(oov_tokens))
print("Test documents with no in-vocabulary tokens:", empty_test_documents)
print("Vocabulary built from training data only:", True)
assert X_train.shape[1] == X_test.shape[1] == len(vocabulary)
display(
    pd.DataFrame(X_train[:5], columns=vocabulary).iloc[
        :, : min(10, len(vocabulary))
    ]
)

## 6. Naive Bayes implemented from scratch

This is a Multinomial Naive Bayes classifier implemented using NumPy and Python—not scikit-learn's Naive Bayes. The implementation follows the four steps from Section 3:

1. **Priors:** `P(c) = N_c / N`.
2. **Vocabulary and denominators:** build the vocabulary from training data and compute `Σ_w count(w, c) + α|V|` for each class.
3. **Likelihoods:** `P(w | c) = (count(w, c) + α) / (Σ_w count(w, c) + α|V|)` with Laplace smoothing (`α=1`) to avoid zero probabilities.
4. **Scores:** add the log prior and the weighted log likelihoods, then predict the class with the highest score.

Log probabilities are used for numerical stability.

In [ ]:
class ManualMultinomialNB:
    """Multinomial Naive Bayes trained with maximum-likelihood counts."""

    def __init__(self, alpha=1.0):
        # Laplace/Lidstone smoothing strength.
        self.alpha = alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_samples, n_features = X.shape
        self.class_log_prior_ = {}
        self.feature_log_prob_ = {}
        self.class_count_ = {}
        self.feature_total_ = {}
        self.smoothing_denominator_ = {}

        for class_label in self.classes_:
            X_class = X[y == class_label]
            class_count = X_class.shape[0]

            # Step 1: empirical class prior, stored in log space.
            self.class_log_prior_[class_label] = np.log(class_count / n_samples)
            self.class_count_[class_label] = class_count

            # Step 2: per-class word totals and the smoothed denominator.
            word_counts = X_class.sum(axis=0).astype(float)
            self.feature_total_[class_label] = word_counts.sum()
            denominator = word_counts.sum() + self.alpha * n_features
            self.smoothing_denominator_[class_label] = denominator

            # Step 3: Laplace-smoothed word likelihoods in log space.
            smoothed_counts = word_counts + self.alpha
            self.feature_log_prob_[class_label] = np.log(
                smoothed_counts / denominator
            )

        return self

    def predict(self, X):
        predictions = []
        for row in X:
            class_scores = []
            for class_label in self.classes_:
                # log P(class) + sum_j count_j * log P(word_j | class)
                score = self.class_log_prior_[class_label]
                score += np.dot(row, self.feature_log_prob_[class_label])
                class_scores.append(score)
            predictions.append(self.classes_[int(np.argmax(class_scores))])
        return np.asarray(predictions)


manual_model = ManualMultinomialNB(alpha=1.0)
manual_model.fit(X_train, y_train)

manual_train_pred = manual_model.predict(X_train)
manual_test_pred = manual_model.predict(X_test)

print("Manual model trained.")
print("Example test predictions:", manual_test_pred[:10])

# Audit the values learned in Steps 1-2 (the section's prior and denominator table).
print("\nManual NB audit (training set):")
print(f"{'Class':<14}{'Prior':>8}{'Tokens':>9}{'Denominator':>13}")
for class_label in manual_model.classes_:
    print(
        f"{class_label:<14}"
        f"{np.exp(manual_model.class_log_prior_[class_label]):>8.4f}"
        f"{int(manual_model.feature_total_[class_label]):>9}"
        f"{manual_model.smoothing_denominator_[class_label]:>13.0f}"
    )

## 7. Train scikit-learn model and compare predictions

Use `MultinomialNB` with the same Bag of Words matrices and the same `alpha=1.0` smoothing setting.

In [ ]:
sklearn_model = MultinomialNB(alpha=1.0)
sklearn_model.fit(X_train, y_train)

sklearn_train_pred = sklearn_model.predict(X_train)
sklearn_test_pred = sklearn_model.predict(X_test)

test_predictions_match = np.mean(manual_test_pred == sklearn_test_pred)
train_predictions_match = np.mean(manual_train_pred == sklearn_train_pred)

print(f"Train predictions matching: {train_predictions_match:.2%}")
print(f"Test predictions matching:  {test_predictions_match:.2%}")

mismatches = test_df.loc[
    manual_test_pred != sklearn_test_pred, ["text", "label"]
].copy()
mismatches["manual_prediction"] = manual_test_pred[
    manual_test_pred != sklearn_test_pred
]
mismatches["sklearn_prediction"] = sklearn_test_pred[
    manual_test_pred != sklearn_test_pred
]

if mismatches.empty:
    print("The two models made identical predictions on the test set.")
    print(
        "This is expected: both use the same BoW matrices, empirical class priors, "
        "Laplace smoothing (alpha=1), and log-space scoring."
    )
else:
    print("Examples where predictions differ:")
    display(mismatches.head(10))
    print(
        "Possible causes include implementation details, class-prior handling, "
        "feature ordering, or numerical tie-breaking."
    )

## 8. Evaluation

Report train and test accuracy for both models, then show a test-set confusion matrix as a heatmap with counts and precision, recall, and F1-score for each class.

In [ ]:
results = pd.DataFrame(
    [
        {
            "Model": "Manual Naive Bayes",
            "Train Accuracy": accuracy_score(y_train, manual_train_pred),
            "Test Accuracy": accuracy_score(y_test, manual_test_pred),
        },
        {
            "Model": "scikit-learn MultinomialNB",
            "Train Accuracy": accuracy_score(y_train, sklearn_train_pred),
            "Test Accuracy": accuracy_score(y_test, sklearn_test_pred),
        },
    ]
)

display(
    results.style.format(
        {
            "Train Accuracy": "{:.4f}",
            "Test Accuracy": "{:.4f}",
        }
    )
)

# Macro and weighted averages complement the per-class report below.
summary = []
for model_name, predictions in [
    ("Manual Naive Bayes", manual_test_pred),
    ("scikit-learn MultinomialNB", sklearn_test_pred),
]:
    macro = precision_recall_fscore_support(
        y_test, predictions, average="macro", zero_division=0
    )
    weighted = precision_recall_fscore_support(
        y_test, predictions, average="weighted", zero_division=0
    )
    summary.append(
        {
            "Model": model_name,
            "Macro Precision": macro[0],
            "Macro Recall": macro[1],
            "Macro F1": macro[2],
            "Weighted Precision": weighted[0],
            "Weighted Recall": weighted[1],
            "Weighted F1": weighted[2],
        }
    )

display(
    pd.DataFrame(summary).style.format(
        {
            "Macro Precision": "{:.4f}",
            "Macro Recall": "{:.4f}",
            "Macro F1": "{:.4f}",
            "Weighted Precision": "{:.4f}",
            "Weighted Recall": "{:.4f}",
            "Weighted F1": "{:.4f}",
        }
    )
)

In [ ]:
# Use one fixed, sorted label order for both confusion matrices.
labels = sorted(np.unique(np.concatenate([y_test, manual_test_pred, sklearn_test_pred])))

# Confusion matrix for the manual model
cm_manual = confusion_matrix(y_test, manual_test_pred, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_manual,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
)
plt.title("Confusion Matrix — Manual Naive Bayes (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# Confusion matrix for scikit-learn
cm_sklearn = confusion_matrix(y_test, sklearn_test_pred, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_sklearn,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=labels,
    yticklabels=labels,
)
plt.title("Confusion Matrix — scikit-learn MultinomialNB (Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
print("Manual Naive Bayes — per-class metrics (test set)")
print(
    classification_report(
        y_test, manual_test_pred, labels=labels, zero_division=0
    )
)

print("scikit-learn MultinomialNB — per-class metrics (test set)")
print(
    classification_report(
        y_test, sklearn_test_pred, labels=labels, zero_division=0
    )
)

## 9. Conclusion

Both implementations achieved **100% training accuracy** and **82.35% test accuracy**
(14 correct predictions out of 17 test examples). The manual Multinomial Naive Bayes
implementation and scikit-learn's `MultinomialNB` produced identical predictions on
the test set, showing that the manual implementation matches the library implementation
on this split.

The class-level F1-scores were:
- **Politics: 0.89** — best-performing class; recall was 1.00.
- **Entertainment: 0.86**.
- **Sports: 0.80**.
- **Technology: 0.77** — lowest F1-score, so it was the hardest class to classify in this test set.

The weighted-average F1-score was **0.82**, while the macro-average F1-score was **0.83**.
The gap between perfect training accuracy and lower test accuracy may indicate
overfitting, although the small dataset makes this conclusion uncertain. The dataset
contains only 85 examples, with 17 in the test set, so these metrics should be
interpreted cautiously. A larger, more balanced dataset and cross-validation would
provide a more reliable estimate of generalization.

The vocabulary, word frequencies, and word clouds were built from the training split
only; the test split was used solely for final evaluation. A two-page summary report
is included in `report/report.pdf`.

## 10. Reproducibility

- The split uses `random_state=42` and stratification, so the same rows are assigned to
  train and test on every run.
- The vocabulary, word frequencies, and word clouds use training data only.
- The notebook is deterministic: the word-cloud layout also receives `random_state=42`.
- Execute it locally with:

  `python -m jupyter nbconvert --to notebook --execute assignment_1.ipynb --output assignment_1_executed.ipynb`

- The two-page report is in `report/report.pdf`.